# Fetal Vein Segmentation — Operational entry point

**Type:** orchestration notebook (Category C).

Open this notebook first. It documents the full experiment lifecycle, the dependency graph between pipeline notebooks, and optional `%run` shortcuts. It does **not** implement scientific algorithms.

---

## 1. Purpose

### Project objective

This repository implements a **notebook-based biomedical image pipeline** for fetal vein segmentation in ultrasound. The workflow compares an original image set and five preprocessing variants (PP1–PP5), trains a UNet per dataset configuration, exports prediction masks, and evaluates performance with standard segmentation metrics (Dice, accuracy, precision, recall), including morphological post-processing aligned with course reference material.

### Repository architecture

```text
02_dataset/          Data (images, labels, preprocessed sets, masks, models)
03_pipeline/         Scientific notebooks (this folder)
04_pipeline_results/ Aggregated evaluation tables (CSV)
06_documentation/    Governance, architecture, MkDocs site
99_system/           Design history and tooling
```

Pipeline layout:

```text
03_pipeline/
├── entrypoint.ipynb          ← you are here (orchestration only)
├── 01_preprocessing/
│   ├── 00_common/            Category A libraries
│   └── 01–05_*_pipeline.ipynb Category B execution (dataset generation)
├── 02_segmentation/          Category B execution (training + masks)
├── 03_postprocessing/        Category A library (metrics, morphology)
└── 04_evaluation/            Category B execution (comparison table)
```

### Notebook taxonomy

| Category | Role | Produces experiment artefacts? |
|----------|------|--------------------------------|
| **A — Library** | Reusable functions loaded with `%run` | No (when imported only) |
| **B — Execution** | Complete processing stage | Yes |
| **C — Orchestration** | Workflow guide and shortcuts | No |

| Kind | Notebooks | What they do |
|------|-----------|--------------|
| **Libraries** | `00_generic`, `01_global_operations`, `02_filtering`, `postprocessing_common` | Define functions; do not run as standalone experiments |
| **Dataset generation** | `01`–`05_preprocessing_pipeline` | Read `images/`, write `images_pp_1` … `images_pp_5/` |
| **Model training** | `fetal_vein_segmentation` | Train UNet, save `.pth`, write `results_*` masks |
| **Evaluation** | `evaluation` | Metrics table → `04_pipeline_results/` |

### Execution philosophy

1. **Single source of science** — All algorithms live in library and execution notebooks; the entry point only orchestrates and documents.
2. **Explicit dependencies** — Execution notebooks `%run` libraries; the graph below shows load order.
3. **Repository-relative paths** — Run from the repository root (folder containing `02_dataset/` and `03_pipeline/`).
4. **Colab-first** — Intended for Google Colab; install dependencies once from root `requirements.txt`.

Further reading: `06_documentation/01_governance/scientific_notebook_standards.md`, `99_system/design_history.md`.

### Environment setup (Google Colab)

1. Clone or upload this repository to Colab.
2. Set the working directory to the **repository root**.
3. Install dependencies: `pip install -r requirements.txt`.

Optional setup (run in a code cell before loading libraries):

```python
%cd /content/fetal_vein_segmentation  # adjust path to your clone
!pip install -q -r requirements.txt
%cd 03_pipeline  # recommended when using relative %run paths below
```

### Dataset layout (`02_dataset/`)

| Directory | Role |
|-----------|------|
| `images/` | Original ultrasound images (read-only input) |
| `labels/` | Ground-truth masks (read-only) |
| `images_pp_1` … `images_pp_5/` | Preprocessed images (pipelines 01–05) |
| `results_original/`, `results_pp_1` … `results_pp_5/` | Predicted masks from segmentation |
| `Save_Models/` | Best model checkpoints (`.pth`) per experiment |

Pairing rule: preprocessed filenames retain the patient identifier (e.g. `P080_IMG1_PP_PL_1.png` → label `P080_IMG1.png`). Implemented in `postprocessing_common.ipynb`.

## 2. Dependency graph

### Pipeline flow (data and notebooks)

```text
00_generic.ipynb
        ↓
01_global_operations.ipynb
        ↓
02_filtering.ipynb
        ↓
Preprocessing Pipelines (01 → 05)
        ↓
Segmentation (fetal_vein_segmentation.ipynb)
        ↓
Post-processing library (postprocessing_common.ipynb)
        ↓
Evaluation (evaluation.ipynb)
```

```mermaid
flowchart TD
    G[00_generic.ipynb] --> GO[01_global_operations.ipynb]
    GO --> F[02_filtering.ipynb]
    F --> P1[01_preprocessing_pipeline]
    F --> P2[02_preprocessing_pipeline]
    F --> P3[03_preprocessing_pipeline]
    F --> P4[04_preprocessing_pipeline]
    F --> P5[05_preprocessing_pipeline]
    P1 & P2 & P3 & P4 & P5 --> DS[(02_dataset/images_pp_*)]
    DS --> SEG[fetal_vein_segmentation.ipynb]
    SEG --> RES[(02_dataset/results_*)]
    PP[postprocessing_common.ipynb] --> EV[evaluation.ipynb]
    RES --> EV
    EV --> CSV[(04_pipeline_results/)]
```

### Component responsibilities

| Component | Path | Responsibility |
|-----------|------|----------------|
| **Generic helpers** | `01_preprocessing/00_common/00_generic.ipynb` | Project root detection, grayscale/`uint8` preparation, image load/save helpers, metadata printing, side-by-side display |
| **Global operations** | `01_preprocessing/00_common/01_global_operations.ipynb` | Point operations: brightness/contrast, inversion, gamma, log, exponential, histogram equalisation |
| **Spatial filters** | `01_preprocessing/00_common/02_filtering.ipynb` | Convolution and filters: average, median, Gaussian, Sobel, Laplacian |
| **Preprocessing pipelines** | `01_preprocessing/0N_preprocessing_pipeline.ipynb` | Batch jobs: read `images/`, apply configured globals + one filter, write `images_pp_N/` |
| **Segmentation** | `02_segmentation/fetal_vein_segmentation.ipynb` | MONAI UNet training, checkpoint export, mask inference to `results_*` |
| **Post-processing library** | `03_postprocessing/postprocessing_common.ipynb` | Morphological opening, largest connected component, `calculate_metrics`, label pairing, path validation |
| **Evaluation** | `04_evaluation/evaluation.ipynb` | Compare Original and PP1–PP5 with/without `pos_process()`; write CSV |

Execution notebooks **embed** the library chain via `%run` internally. The entry point can preload libraries once (Section 3) for interactive exploration; pipeline notebooks still load what they need if run standalone.

## 3. Common library loader

Run the code cell below to import shared functions into the current kernel. This is **optional** before running execution notebooks; it is useful for inspection, path validation, or calling helpers from this notebook.

### `00_generic.ipynb`

- Image loading from disk paths
- Metadata utilities (`print_image_metadata`)
- Visualisation helpers (`display_images_side_by_side`)
- Project root detection (`find_project_root`)
- Grayscale and `uint8` preparation helpers

### `01_global_operations.ipynb`

- Brightness and contrast (linear transform)
- Gamma correction
- Logarithmic and exponential transforms
- Histogram computation and equalisation
- Intensity inversion

### `02_filtering.ipynb`

- Average filter
- Median filter
- Gaussian filter
- Sobel filter
- Laplacian filter
- Manual convolution helper (shared by filters)

### `postprocessing_common.ipynb`

- Morphological post-processing (`pos_process`: opening + largest connected component)
- Connected-component labelling (inside `pos_process`)
- Segmentation metrics (`calculate_metrics`: Dice, accuracy, precision, recall)
- Mask load/binarise/orientation helpers
- Identifier-based label pairing (`resolve_label_path`)

### Important

This step **only loads functions** into memory.

- No images are processed.
- No models are trained.
- No files are generated.

To produce datasets, train models, or write results, use Sections 4–6.

In [ ]:
# Load preprocessing libraries

%run ./01_preprocessing/00_common/00_generic.ipynb
%run ./01_preprocessing/00_common/01_global_operations.ipynb
%run ./01_preprocessing/00_common/02_filtering.ipynb

# Load post-processing library

%run ./03_postprocessing/postprocessing_common.ipynb

print("Common libraries loaded successfully.")

## 4. Preprocessing execution section

Preprocessing notebooks are **Category B execution** notebooks. Each generates one preprocessed dataset under `02_dataset/`.

| Pipeline | Notebook | Filter | Output folder |
|----------|----------|--------|---------------|
| **PP1** | `01_preprocessing/01_preprocessing_pipeline.ipynb` | Average filter | `02_dataset/images_pp_1/` |
| **PP2** | `01_preprocessing/02_preprocessing_pipeline.ipynb` | Median filter | `02_dataset/images_pp_2/` |
| **PP3** | `01_preprocessing/03_preprocessing_pipeline.ipynb` | Gaussian filter | `02_dataset/images_pp_3/` |
| **PP4** | `01_preprocessing/04_preprocessing_pipeline.ipynb` | Sobel filter | `02_dataset/images_pp_4/` |
| **PP5** | `01_preprocessing/05_preprocessing_pipeline.ipynb` | Laplacian filter | `02_dataset/images_pp_5/` |

**Input (all pipelines):**

```text
02_dataset/images/
```

**Outputs:**

```text
02_dataset/images_pp_1/
02_dataset/images_pp_2/
02_dataset/images_pp_3/
02_dataset/images_pp_4/
02_dataset/images_pp_5/
```

Each notebook loads the three preprocessing libraries, applies pipeline-specific global operations (configured in that notebook), then applies its spatial filter. Filenames use the course suffix pattern `_PP_PL_N`.

Run the next cell to execute one or more pipelines. **Uncomment** only the pipelines you need (each run may take several minutes).

In [ ]:
# Optional: run preprocessing pipelines (uncomment to execute)

# Pipeline 1 — Average filter → images_pp_1/
# %run ./01_preprocessing/01_preprocessing_pipeline.ipynb

# Pipeline 2 — Median filter → images_pp_2/
# %run ./01_preprocessing/02_preprocessing_pipeline.ipynb

# Pipeline 3 — Gaussian filter → images_pp_3/
# %run ./01_preprocessing/03_preprocessing_pipeline.ipynb

# Pipeline 4 — Sobel filter → images_pp_4/
# %run ./01_preprocessing/04_preprocessing_pipeline.ipynb

# Pipeline 5 — Laplacian filter → images_pp_5/
# %run ./01_preprocessing/05_preprocessing_pipeline.ipynb

print("Preprocessing shortcuts idle. Uncomment a %run line above to generate a dataset.")

## 5. Segmentation execution section

**Notebook:** `02_segmentation/fetal_vein_segmentation.ipynb`

This **Category B execution** notebook:

- trains a **UNet** (MONAI);
- saves the best checkpoint under `02_dataset/Save_Models/`;
- generates **prediction masks** under the configured `results_*` folder.

### Valid training datasets

Set `DATASET_FOLDER` and matching `RESULTS_FOLDER` / `MODEL_NAME` in the segmentation configuration cell for each experiment:

| Experiment | `DATASET_FOLDER` | `RESULTS_FOLDER` |
|------------|------------------|------------------|
| Original | `images` | `results_original` |
| PP1 | `images_pp_1` | `results_pp_1` |
| PP2 | `images_pp_2` | `results_pp_2` |
| PP3 | `images_pp_3` | `results_pp_3` |
| PP4 | `images_pp_4` | `results_pp_4` |
| PP5 | `images_pp_5` | `results_pp_5` |

All six are valid experiment datasets. Run the notebook **once per row**, changing only the configuration cell between runs.

The notebook does **not** produce the final course comparison table; that is the role of evaluation (Section 6).

Optional shortcut: uncomment the line below to run the segmentation notebook from here (ensure configuration inside that notebook matches your target experiment).

In [ ]:
# Optional: run segmentation (edit DATASET_FOLDER / RESULTS_FOLDER / MODEL_NAME in that notebook first)

# %run ./02_segmentation/fetal_vein_segmentation.ipynb

print("Segmentation shortcut idle. Open fetal_vein_segmentation.ipynb or uncomment %run above.")

## 6. Evaluation section

**Notebook:** `04_evaluation/evaluation.ipynb`

This **Category B execution** notebook computes:

- **Dice** coefficient
- **Accuracy**
- **Precision**
- **Recall**

Metrics are computed **with and without** morphological post-processing (`pos_process`), using functions from `postprocessing_common.ipynb`.

### Comparison matrix

Evaluation compares the following conditions (each experiment folder under `02_dataset/results_*`):

| Experiment | Without post-processing | With post-processing |
|------------|-------------------------|----------------------|
| **Original** | Original | Original + Post-processing |
| **PP1** | PP1 | PP1 + Post-processing |
| **PP2** | PP2 | PP2 + Post-processing |
| **PP3** | PP3 | PP3 + Post-processing |
| **PP4** | PP4 | PP4 + Post-processing |
| **PP5** | PP5 | PP5 + Post-processing |

**Output:** aggregated table saved to:

```text
04_pipeline_results/tabela_avaliacao_experiencias.csv
```

Prerequisites: prediction masks exist for each experiment you include; labels present in `02_dataset/labels/`.

Optional shortcut: uncomment the cell below to run evaluation from here.

In [ ]:
# Optional: run full evaluation notebook

# %run ./04_evaluation/evaluation.ipynb

print("Evaluation shortcut idle. Uncomment %run above after segmentation outputs exist.")

## 7. Recommended workflow

Official execution order for a full comparative study:

```text
1. Load common libraries          (Section 3 — optional but recommended)
2. Generate preprocessing datasets (Section 4 — pipelines 01–05)
3. Train segmentation model         (Section 5 — once per dataset: Original + PP1–PP5)
4. Generate prediction masks      (same notebook run — written to results_*)
5. Apply post-processing            (inside evaluation — pos_process)
6. Run evaluation                   (Section 6)
7. Analyse metrics                  (CSV + tables in evaluation notebook)
8. Select best-performing pipeline  (compare Dice and secondary metrics)
```

### End-to-end path (repository level)

```text
GitHub repository
        ↓
   Google Colab (requirements.txt)
        ↓
     02_dataset/          ← images, labels, pp_*, results_*, models
        ↓
     03_pipeline/         ← this workflow
        ↓
04_pipeline_results/       ← evaluation CSV
        ↓
     05_report/            ← academic report (outside pipeline code)
```

### Quick reference — what to open

| Step | Open / run |
|------|------------|
| Start | `entrypoint.ipynb` (this file) |
| Libraries | Section 3 code cell, or rely on per-notebook `%run` |
| PP1–PP5 | `01_preprocessing/0N_preprocessing_pipeline.ipynb` |
| Train + infer | `02_segmentation/fetal_vein_segmentation.ipynb` |
| Metrics | `04_evaluation/evaluation.ipynb` |

## 8. Architectural notes

### Libraries vs experiments

- **Common notebooks are libraries.** They define reusable functions and are imported with `%run`. They are not experiment entry points and must not be treated as complete studies on their own.
- **Execution notebooks are experiments.** They read and write under `02_dataset/` (and evaluation writes under `04_pipeline_results/`). They encode one stage of the scientific pipeline.

### Role of this entry point

- The **entrypoint notebook exists only to orchestrate execution and document dependencies.**
- It must **never duplicate scientific implementations.** All algorithms remain in the original library and execution notebooks.
- Markdown here may summarise behaviour; authoritative code and formulas stay in Category A/B notebooks.

### What this notebook does not do

- Does not modify preprocessing mathematics, filter kernels, or global operation formulas.
- Does not change MONAI/UNet configuration, training loops, or loss definitions.
- Does not alter `pos_process` / `calculate_metrics` logic.
- Does not rename dataset folders or change output CSV schema.

### Documentation map

| Topic | Location |
|-------|----------|
| Dataset layout | `02_dataset/README.md` |
| Pipeline index | `03_pipeline/README.md` |
| Execution workflow | `06_documentation/02_architecture/execution_workflow.md` |
| Data flow | `06_documentation/02_architecture/data_flow.md` |
| Governance v6 | `06_documentation/01_governance/project_governance.md` |
| Root overview | `README.md` |